In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [50]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [51]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [52]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [53]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [54]:
import json


def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [55]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered late can I join enrollment late course FAQ"}
function_call: search {"query":"course registration late join discovered the course can I join"}
function_call: search {"query":"enrollment deadline late registration join course FAQ"}


In [56]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enrollment late course FAQ"}', call_id='call_JP10aCoPmdNuXd0SlnMIGzPo', name='search', type='function_call', id='fc_0a1b748a5a2913e7006a37617e298c8195b33206b1c1efc0b8', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course registration late join discovered the course can I join"}', call_id='ca

In [57]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join and follow along. The course materials are available, so you can start anytime.

One important caveat: if you want a certificate, you need to submit your project while submissions are still being accepted. Certificates aren’t available for self-paced completion after the live cohort ends.

If you want, I can also help you figure out how to start the course or what the weekly workflow looks like.


In [58]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [59]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally Ollama local installation FAQ"}
function_call: search {"query":"Ollama local run model start service FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed, start it locally, and open a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"rol

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed, start it locally, and open a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restarting the serve

In [60]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"can I still join the course after it started discovered the course join late enrollment FAQ"}
function_call: search {"query":"late enrollment course join after start discovered the course FAQ"}
function_call: search {"query":"course enrollment open can I join anytime FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

If your goal is a certificate, the key requirement is that you submit your project while submissions are still open. You can also start learning anytime because the videos and materials are available.

If you want, I can also help you find:
- where to start in the course
- the homework/project workflow
- certificate requirements

Anything else you’d like to explore?


'Yes — you can still join the course even if you just discovered it.\n\nIf your goal is a certificate, the key requirement is that you submit your project while submissions are still open. You can also start learning anytime because the videos and materials are available.\n\nIf you want, I can also help you find:\n- where to start in the course\n- the homework/project workflow\n- certificate requirements\n\nAnything else you’d like to explore?'

In [61]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"discover the course can I join it enrollment join late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. If you want a certificate, make sure you submit your project while submissions are still open.

You can also start learning right away; registration isn’t required just to begin the materials.

Would you like help with how to get started or how the course workflow works?


'Yes — you can still join the course. If you want a certificate, make sure you submit your project while submissions are still open.\n\nYou can also start learning right away; registration isn’t required just to begin the materials.\n\nWould you like help with how to get started or how the course workflow works?'

In [62]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit opening chess"}
iteration #2...
function_call: search {"query":"queen gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. `d4 d5`
2. `c4`

White offers the c-pawn to try to draw Black’s **d-pawn** away from the center and gain control of the middle of the board.

A common continuation is:

- `1. d4 d5 2. c4`

If Black accepts, it’s called the **Queen’s Gambit Accepted**. If Black doesn’t, it’s the **Queen’s Gambit Declined**.

If you meant **the Netflix show/book** instead of the chess opening, I can explain that too. Want a simple explanation of how the opening works?


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. `d4 d5`\n2. `c4`\n\nWhite offers the c-pawn to try to draw Black’s **d-pawn** away from the center and gain control of the middle of the board.\n\nA common continuation is:\n\n- `1. d4 d5 2. c4`\n\nIf Black accepts, it’s called the **Queen’s Gambit Accepted**. If Black doesn’t, it’s the **Queen’s Gambit Declined**.\n\nIf you meant **the Netflix show/book** instead of the chess opening, I can explain that too. Want a simple explanation of how the opening works?'

In [63]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen’s gambit,” so I can’t answer that from the course database.

If you meant something specific in the course, please rephrase with the course context, and I can look again. Are there other areas you want to explore?


'I couldn’t find a course-related FAQ entry for “queen’s gambit,” so I can’t answer that from the course database.\n\nIf you meant something specific in the course, please rephrase with the course context, and I can look again. Are there other areas you want to explore?'